In [1]:
# Assignment 3
## RAG

In [2]:
!pip install -q -U pocketflow

In [3]:
%pip install -q -U "google-genai>=1.0.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 732.2/732.2 kB 23.9 MB/s eta 0:00:00


In [4]:
!pip install -q -U faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 18.3 MB/s eta 0:00:00


In [5]:
# Setup your API key
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

In [6]:
# 1. Fixed-Size Chunking
def fixed_size_chunk(text, chunk_size=50):
    chunks = []
    for start_index in range(0, len(text), chunk_size):
        chunks.append(text[start_index:start_index+chunk_size])
    return chunks

In [7]:
# Call the embed_content method with the model to generate text embeddings.
import numpy as np
from google import genai

client = genai.Client(api_key=GEMINI_API_KEY)

def get_embedding(text):

    result = client.models.embed_content(
        model="gemini-embedding-001",
        contents=text
    )

    # Extract embedding vector
    embedding = result.embeddings[0].values

    # Convert to numpy array
    return np.array(embedding, dtype=np.float32)

In [8]:
import faiss

In [9]:
from pocketflow import Node, Flow, BatchNode

In [10]:
texts = [
    """Pocket Flow is a 100-line minimalist LLM framework.
Lightweight: Just 100 lines. Zero bloat, zero dependencies, zero vendor lock-in.
Expressive: Everything you love — Multi-Agents, Workflow, RAG, and more.
Agentic Coding: Let AI Agents (e.g., Cursor AI) build Agents — 10x productivity boost!
To install, pip install pocketflow or just copy the source code (only 100 lines).""",

    """PocketFlow allows developers to build intelligent workflows easily.
It connects multiple AI steps such as chunking, embedding, retrieval, and generation.
This makes it ideal for building RAG applications, chatbots, and knowledge assistants.
The framework focuses on clarity and simplicity while maintaining powerful capabilities.""",

    """In semantic search systems, documents are first divided into smaller chunks.
Each chunk is converted into embeddings which represent the meaning of the text.
These embeddings are stored inside a vector database such as FAISS.
During search, the query is embedded and compared with stored embeddings to find similar content.""",

    """Field tests successfully remediated previously permanently contaminated sites.
Deployment costs 80% less than traditional chemical extraction methods.
Such AI-driven workflows improve efficiency, reduce cost, and accelerate innovation.
This demonstrates how lightweight AI frameworks can solve real-world problems."""
]

In [11]:
query = "What is Semantic Search?"

In [12]:
shared = {
    "texts": texts,
    "embeddings": None,
    "index": None,
    "query": query,
    "query_embedding": None,
    "retrieved_document": None,
}

In [13]:
class ChunkDocumentsNode(BatchNode):

    def prep(self, shared):
        # Fetch original texts from storage
        return shared["texts"]

    def exec(self, text):
        # Break one document into smaller parts
        return fixed_size_chunk(text)

    def post(self, shared, prep_res, exec_res_list):
        # Save all chunks back to storage

        # Merge nested chunk lists into one list
        all_chunks = []
        for chunks in exec_res_list:
            all_chunks.extend(chunks)

        # Update stored texts with chunked versions
        shared["texts"] = all_chunks

        print(f" Created {len(all_chunks)} chunks from {len(prep_res)} documents")

        return "default"

In [14]:
# Nodes for offline flow
class EmbedDocumentsNode(BatchNode):
    def prep(self, shared):
        """Fetch document texts from shared storage"""
        return shared["texts"]

    def exec(self, text):
        """Generate embedding vector for one text"""
        return get_embedding(text)

    def post(self, shared, prep_res, exec_res_list):
        """Save all produced embeddings back to shared storage"""
        embeddings = np.array(exec_res_list, dtype=np.float32)
        shared["embeddings"] = embeddings
        print(f" Created {len(embeddings)} document vectors")
        return "default"



 # Nodes for online flow
class EmbedQueryNode(Node):
    def prep(self, shared):
        """Obtain user question from shared dictionary"""
        return shared["query"]

    def exec(self, query):
        """Convert query text into embedding"""
        print(f" Processing query embedding: {query}")
        query_embedding = get_embedding(query)
        return np.array([query_embedding], dtype=np.float32)

    def post(self, shared, prep_res, exec_res):
        """Store query embedding for retrieval step"""
        shared["query_embedding"] = exec_res
        return "default"

In [15]:
# Nodes for offline flow
class CreateIndexNode(Node):
    def prep(self, shared):
        """Retrieve stored embeddings from shared memory"""
        return shared["embeddings"]

    def exec(self, embeddings):
        """Initialize FAISS index and insert vectors"""
        print(" Building vector search index...")
        dimension = embeddings.shape[1]

        # Initialize flat L2 FAISS index
        index = faiss.IndexFlatL2(dimension)

        # Insert embedding vectors into index
        index.add(embeddings)

        return index

    def post(self, shared, prep_res, exec_res):
        """Save constructed index for later retrieval"""
        shared["index"] = exec_res
        print(f" Search index ready with {exec_res.ntotal} entries")
        return "default"

In [16]:
# Nodes for online flow
class RetrieveDocumentNode(Node):
    def prep(self, shared):
        """Collect query vector, index and stored texts"""
        return shared["query_embedding"], shared["index"], shared["texts"]

    def exec(self, inputs):
        """Perform similarity search inside FAISS index"""
        print(" Looking for nearest document...")
        query_embedding, index, texts = inputs

        # Run vector similarity search
        distances, indices = index.search(query_embedding, k=3)

        # Identify best matching position
        best_idx = indices[0][0]
        distance = distances[0][0]

        # Retrieve matching text chunk
        retrieved_chunks = [texts[i] for i in indices[0]]
        most_relevant_text = " ".join(retrieved_chunks)

        return {
            "text": most_relevant_text,
            "index": best_idx,
            "distance": distance
        }

    def post(self, shared, prep_res, exec_res):
        """Store retrieved result for answer generation"""
        shared["retrieved_document"] = exec_res
        print(f" Retrieved index: {exec_res['index']} | distance: {exec_res['distance']:.4f}")
        print(f" Best text: {exec_res['text']}")
        return "default"

In [17]:
# Create offline flow

chunk_docs_node = ChunkDocumentsNode()
embed_docs_node = EmbedDocumentsNode()
create_index_node = CreateIndexNode()

# Link steps together
chunk_docs_node >> embed_docs_node >> create_index_node

# Create flow starting from chunking
offline_flow = Flow(start=chunk_docs_node)

# Execute once to prepare vector index
# offline_flow.run(shared)

In [18]:
# Create online flow

embed_query_node = EmbedQueryNode()
retrieve_doc_node = RetrieveDocumentNode()

# Link query embedding to retrieval
embed_query_node >> retrieve_doc_node

# Create flow starting from query embedding
online_flow = Flow(start=embed_query_node)

# Run each time user asks something
# online_flow.run(shared)

In [19]:
# Run the offline flow
offline_flow.run(shared)

 Created 29 chunks from 4 documents
 Created 29 document vectors
 Building vector search index...
 Search index ready with 29 entries


'default'

In [20]:
# Run the online flow
online_flow.run(shared)

 Processing query embedding: What is Semantic Search?
 Looking for nearest document...
 Retrieved index: 15 | distance: 0.3990
 Best text: In semantic search systems, documents are first di r database such as FAISS.
During search, the query  into embeddings which represent the meaning of th


'default'

In [21]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 8.8 MB/s eta 0:00:00


In [22]:
from groq import Groq

In [23]:
groq_client = Groq(api_key="your_key_here")

In [24]:
# Get retrieved text from semantic search
context = shared["retrieved_document"]["text"]

print("\nRetrieved Context:\n")
print(context)


Retrieved Context:

In semantic search systems, documents are first di r database such as FAISS.
During search, the query  into embeddings which represent the meaning of th


In [25]:
question = "How does semantic search retrieve relevant documents?"

In [26]:
prompt = f"""
You are a helpful AI assistant.

Answer the question using only the information provided in the context.

Context:
{context}

Question:
{question}

Answer:
"""

In [27]:
#generate
response = groq_client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": prompt}]
)

print("\nGenerated Answer:\n")
print(response.choices[0].message.content)


Generated Answer:

Semantic search retrieves relevant documents by first storing them in a database such as FAISS, and then during search, converting the query into embeddings which represent the meaning of the text.
